# 08 — Hasta dónde crece la ventana, y qué cuesta

**Laboratorio aparte**, no uno de los bloques del taller. Sale de una pregunta
que apareció midiendo la ventana de contexto en el
[notebook 01](01_del_prompt_al_agente.ipynb): ahí subimos `num_ctx` al doble y el
modelo recuperó la clave que antes perdía. Si se puede subir, **¿hasta dónde?**

La respuesta corta es que hay **tres techos distintos**, y el que te detiene no
es el que anuncia el modelo:

| techo | qué lo fija | cómo se ve cuando lo tocas |
|---|---|---|
| el del **modelo** | `qwen35.context_length` = 262 144 | Ollama no te da más |
| el de la **memoria** | cuánta RAM y VRAM hay | falla al cargar, con un error |
| el de la **paciencia** | cuántas capas quedan en la GPU | no falla: se vuelve lento |

El tercero es el que importa en la práctica, y es el único que no avisa.

> **No corras esto durante el taller.** Carga y descarga el modelo siete veces y
> pide hasta 15 GB. Si otras personas están compartiendo tu Ollama, se queda sin
> servicio. Es un laboratorio para antes, en tu propia máquina y con la GPU
> para ti.
>
> Y **las cifras de abajo son de esta máquina** —una RTX 4060 Laptop de 8 GB con
> Ollama en Docker—. En otra GPU la forma de las curvas se repite; los números,
> no.

In [ ]:
# Lo que hay antes de medir: la maquina, el modelo y lo que Ollama declara.
import json
import re
import subprocess
import sys
import time
from pathlib import Path

import requests

sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
#
# Sin esto, basta con que alguien actualice el repo con el kernel abierto para
# que la primera celda falle con un error que no menciona la causa.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")

from agentlib import base_url, modelo_configurado

# Del .env, no escritos aqui: este laboratorio tiene que poder medir el
# modelo y el servidor que esten configurados, no uno fijo.
URL, MODELO = base_url(), modelo_configurado()
DESTINO = Path.cwd().parent / "datos" / "medicion_ventana.json"


def nvidia():
    """Lo que ve el driver. Es la verdad del sistema, no la que reporta Ollama."""
    try:
        s = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().split("\n")[0]
        nombre, total, usado, libre = [x.strip() for x in s.split(",")]
        return {"gpu": nombre, "total_mib": int(total),
                "usado_mib": int(usado), "libre_mib": int(libre)}
    except Exception as e:
        return {"gpu_error": str(e)}   # sin GPU, o nvidia-smi no esta en el PATH


ficha = requests.post(f"{URL}/api/show", json={"model": MODELO}, timeout=120).json()
declarado = next(v for k, v in ficha["model_info"].items()
                 if k.endswith("context_length"))
pesos_gb = next(m["size"] for m in requests.get(f"{URL}/api/tags", timeout=30)
                .json()["models"] if m["name"] == MODELO) / 1e9

g = nvidia()
print(f"  ollama en          : {URL}")
print(f"  modelo             : {MODELO}")
print(f"  ctx que declara    : {declarado:,} tokens")
print(f"  pesos en disco     : {pesos_gb:.2f} GB")
if "gpu" in g:
    print(f"  GPU                : {g['gpu']}")
    print(f"  VRAM               : {g['total_mib']:,} MiB "
          f"({g['libre_mib']:,} libres ahora mismo)")

print(f"\n  Con {pesos_gb:.1f} GB de pesos y {g.get('total_mib', 0)/1024:.1f} GiB de VRAM,")
print(f"  el margen para el KV cache es lo que sobra. Eso es lo que vamos a medir.")


## Qué se mide en cada escalón

Siete tamaños de ventana, cada uno el doble del anterior, de 4 096 hasta los
262 144 que declara el modelo. En cada uno, el ciclo completo:

1. **Descargar el modelo** de memoria con `keep_alive: 0`. Sin esto, el escalón
   nuevo hereda la asignación del anterior y la medición no vale nada.
2. **Cargarlo** pidiendo ese `num_ctx`, y cronometrar.
3. **Preguntarle a `/api/ps`** dónde quedó la memoria.
4. **Preguntarle a `nvidia-smi`** lo mismo, porque las dos fuentes no cuentan lo
   mismo: Ollama reporta *su* modelo, el driver reporta *la GPU entera*.
5. **Repetir la llamada ya cargado**, para separar el costo de cargar del costo
   de responder.

Las métricas que salen de ahí, y por qué cada una:

| métrica | de dónde | qué contesta |
|---|---|---|
| `ctx_concedido` | `/api/ps` | ¿te dio la ventana que pediste, o la recortó? |
| `total_gb` | `/api/ps` | cuánto ocupa todo: pesos + KV cache + buffers |
| `en_gpu_gb` / `en_cpu_gb` | `/api/ps` | **el reparto**: lo que no cabe en VRAM corre en CPU |
| `capas_en_gpu` | el log de Ollama | de 33 capas, cuántas quedaron en la GPU |
| `carga_s` | el reloj | cuánto tarda en estar listo |
| `tokens_por_s` | `eval_count / eval_duration` | la velocidad de generación, ya cargado |

La del reparto es la interesante. **Ollama no falla cuando el modelo no cabe en
la GPU: manda capas a la CPU y sigue.** Por eso el techo de la paciencia no
avisa.

In [ ]:
# Las piezas. Ninguna decide nada: solo preguntan y cronometran.
PROMPT = [{"role": "user", "content": "Escribe exactamente la palabra LISTO y nada mas."}]
LIMITE = 420          # por escalon. Los grandes tardan en asignar la memoria.


def descargar():
    """
    keep_alive 0 saca el modelo de memoria.

    Es el paso que parece de adorno y no lo es: si el modelo sigue cargado con
    la ventana anterior, Ollama reusa esa instancia y el escalon nuevo te
    devuelve los numeros del viejo.
    """
    try:
        requests.post(f"{URL}/api/generate",
                      json={"model": MODELO, "keep_alive": 0}, timeout=120)
    except Exception:
        pass
    time.sleep(3)


def donde_vive():
    """El reparto de memoria del modelo cargado, segun Ollama."""
    cargados = requests.get(f"{URL}/api/ps", timeout=30).json().get("models", [])
    if not cargados:
        return {}
    v = cargados[0]
    total, en_gpu = v.get("size", 0), v.get("size_vram", 0)
    return {"ctx_concedido": v.get("context_length"),
            "total_gb": round(total / 1e9, 2),
            "en_gpu_gb": round(en_gpu / 1e9, 2),
            "en_cpu_gb": round((total - en_gpu) / 1e9, 2),
            "pct_en_gpu": round(100 * en_gpu / total, 1) if total else None}


def una_llamada(num_ctx):
    """Una peticion con esa ventana. Los tiempos salen de Ollama, en nanosegundos."""
    t0 = time.time()
    r = requests.post(f"{URL}/api/chat", json={
        "model": MODELO, "messages": PROMPT, "stream": False, "think": False,
        "options": {"temperature": 0, "num_predict": 16, "num_ctx": num_ctx},
    }, timeout=LIMITE)
    pared = time.time() - t0
    r.raise_for_status()
    d, ns = r.json(), 1e9
    return {"pared_s": round(pared, 1),
            "carga_s": round(d.get("load_duration", 0) / ns, 1),
            "prompt_tokens": d.get("prompt_eval_count"),
            "generados": d.get("eval_count"),
            "tokens_por_s": round(d["eval_count"] / (d["eval_duration"] / ns), 1)
            if d.get("eval_duration") else None,
            "respuesta": d["message"]["content"].strip()[:20]}


def capas_por_ventana():
    """
    Cuantas capas quedaron en la GPU, por tamano de ventana.

    Esto NO esta en la API: sale del log de Ollama, que es el unico que lo dice.
    Requiere que Ollama corra en Docker en esta maquina; si no, se omite y las
    demas metricas siguen igual.
    """
    try:
        log = subprocess.run(["docker", "logs", "agenticai_ollama"],
                             capture_output=True, text=True, timeout=60)
        patron = re.compile(r"Operation:commit.*?KvSize:(\d+).*?GPULayers:(\d+)")
        salida = {}
        for linea in (log.stdout + log.stderr).split("\n"):
            m = patron.search(linea)
            if m:
                salida[int(m.group(1))] = int(m.group(2))   # el ultimo gana
        return salida
    except Exception:
        return {}


## La rampa

**Tarda entre 8 y 10 minutos**, y el último escalón **falla a propósito**: es el
que contesta la pregunta. Mientras corre, la máquina va a estar ocupada — el
escalón de 131 072 pide unos 15 GB entre VRAM y RAM.

El bucle se detiene en el primer fallo. No tiene sentido seguir subiendo después
del techo, y seguir pidiendo memoria que no hay es la forma rápida de dejar la
laptop inservible por un rato.

In [ ]:
# La rampa. Se detiene en el primer escalon que no carga.
RAMPA = (4096, 8192, 16384, 32768, 65536, 131072, 262144)

MEDICIONES = []
for ctx in RAMPA:
    descargar()
    fila = {"num_ctx": ctx}
    try:
        fria = una_llamada(ctx)                    # incluye la carga
        fila.update(donde_vive())
        fila["carga_s"] = fria["pared_s"]
        fila["prompt_tokens"] = fria["prompt_tokens"]
        fila["gpu_usado_mib"] = nvidia().get("usado_mib")
        fila["tokens_por_s"] = una_llamada(ctx)["tokens_por_s"]   # ya cargado
        fila["ok"] = True
        print(f"  {ctx:>7,} -> concedido {fila.get('ctx_concedido'):>7,} | "
              f"total {fila['total_gb']:>5.2f} GB = gpu {fila['en_gpu_gb']:>4.2f} + "
              f"cpu {fila['en_cpu_gb']:>4.2f} | carga {fila['carga_s']:>4.1f}s | "
              f"{fila['tokens_por_s']:>4.1f} tok/s", flush=True)
    except Exception as e:
        fila["ok"] = False
        fila["error"] = f"{type(e).__name__}: {e}"[:120]
        print(f"  {ctx:>7,} -> NO CARGO. {fila['error']}", flush=True)
        MEDICIONES.append(fila)
        break
    MEDICIONES.append(fila)

# Las capas salen del log, y se pegan a la fila que corresponde.
capas = capas_por_ventana()
for fila in MEDICIONES:
    fila["capas_en_gpu"] = capas.get(fila["num_ctx"])

descargar()
DESTINO.write_text(json.dumps(MEDICIONES, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"\n  {len(MEDICIONES)} escalones guardados en datos/{DESTINO.name}")
print(f"  (las graficas leen de ahi, asi que se pueden rehacer sin volver a medir)")


### Todas las métricas, en una tabla

La tabla primero y las gráficas después, a propósito: la gráfica muestra la
forma, la tabla es el dato. Y hay una columna que solo se entiende leyendo las
dos juntas.

In [ ]:
# Todo lo medido, sin resumir.
filas = json.loads(DESTINO.read_text(encoding="utf-8"))

enc = (f"  {'ventana':>8} {'concedido':>9} {'total':>7} {'en GPU':>7} {'en CPU':>7} "
       f"{'% GPU':>6} {'capas':>6} {'carga':>7} {'tok/s':>6}")
print(enc)
print("  " + "-" * (len(enc) - 2))
for f in filas:
    if not f.get("ok"):
        print(f"  {f['num_ctx']:>8,} {'NO CARGO':>9}   <- el techo: {f['error'][:44]}")
        continue
    print(f"  {f['num_ctx']:>8,} {f['ctx_concedido']:>9,} {f['total_gb']:>7.2f} "
          f"{f['en_gpu_gb']:>7.2f} {f['en_cpu_gb']:>7.2f} {f['pct_en_gpu']:>5.0f}% "
          f"{(f.get('capas_en_gpu') or '--'):>6} {f['carga_s']:>6.1f}s "
          f"{f['tokens_por_s']:>6.1f}")

ok = [f for f in filas if f.get("ok")]
if len(ok) > 1:
    a, b = ok[0], ok[-1]
    crecio = (b["total_gb"] - a["total_gb"]) * 1e9 / (b["num_ctx"] - a["num_ctx"])
    print(f"\n  el KV cache cuesta ~{crecio/1024:.1f} KiB por token de ventana")
    print(f"  (medido de la pendiente entre {a['num_ctx']:,} y {b['num_ctx']:,},")
    print(f"  no de una formula: depende de GQA y de como Ollama cuantice el cache)")
    print(f"  velocidad: de {a['tokens_por_s']} a {b['tokens_por_s']} tok/s "
          f"-> {a['tokens_por_s']/b['tokens_por_s']:.1f} veces mas lento")


### Las mismas cifras, en cuatro gráficas

Cada una contesta una pregunta distinta, y las de abajo explican las de arriba: **las capas en la GPU son la causa y los tokens por segundo el efecto**. Van en gráficas separadas y no en una con dos ejes, que es la forma más rápida de hacer que una correlación parezca una ley.

In [ ]:
# Las cuatro graficas. Leen del JSON, asi que no hay que volver a medir.
import matplotlib.pyplot as plt

filas = json.loads(DESTINO.read_text(encoding="utf-8"))
ok = [f for f in filas if f.get("ok")]
fallo = next((f for f in filas if not f.get("ok")), None)

# Paleta: dos categoricas validadas contra el fondo claro, y la tinta en gris.
AZUL, NARANJA = "#2a78d6", "#eb6834"
TINTA, SUAVE, MUDO = "#0b0b0b", "#52514e", "#898781"
REJILLA, FONDO = "#e1e0d9", "#fcfcfb"

plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"], "font.size": 9,
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": SUAVE, "text.color": TINTA,
    "xtick.color": MUDO, "ytick.color": MUDO, "axes.titlecolor": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": REJILLA, "grid.linewidth": 0.8, "axes.titlesize": 10,
    "axes.titlelocation": "left", "axes.titlepad": 10, "lines.linewidth": 2,
})

x = list(range(len(ok)))
etiquetas = [f"{f['num_ctx']//1024}K" for f in ok]
vram_gb = (nvidia().get("total_mib") or 0) * 1.048576 / 1000   # MiB -> GB decimales

fig, ejes = plt.subplots(2, 2, figsize=(11.5, 7.6))
fig.suptitle("Qué cuesta agrandar la ventana de contexto", x=0.065, y=0.985,
             ha="left", fontsize=13, color=TINTA)
fig.text(0.065, 0.945, f"{MODELO} en una {nvidia().get('gpu', 'GPU')}, "
         f"Ollama en Docker. Cada escalón es el doble del anterior.",
         ha="left", fontsize=9, color=SUAVE)

# --- 1. Donde vive la memoria. Apilada, porque la pregunta es el reparto. ----
ax = ejes[0][0]
ax.set_title("Dónde vive el modelo, en GB")
gpu_v = [f["en_gpu_gb"] for f in ok]
cpu_v = [f["en_cpu_gb"] for f in ok]
ax.bar(x, gpu_v, 0.6, color=AZUL, label="en la GPU", zorder=3)
ax.bar(x, cpu_v, 0.6, bottom=[g + 0.07 for g in gpu_v], color=NARANJA,
       label="en la CPU", zorder=3)        # el 0.07 es el aire entre segmentos
if vram_gb:
    ax.axhline(vram_gb, color=MUDO, lw=0.9, zorder=2)
    ax.text(-0.45, vram_gb + 0.3, f"VRAM de la tarjeta: {vram_gb:.1f} GB",
            ha="left", fontsize=8, color=SUAVE)
if fallo:
    pedido_gb = 12.1 * 1.073741824          # los 12.1 GiB que reporto el log
    ax.plot([len(ok)], [pedido_gb], marker="x", ms=9, mew=2, color=MUDO, zorder=4)
    ax.annotate(f"pidió {pedido_gb:.1f} GB\ny no cargó",
                xy=(len(ok), pedido_gb - 0.4), xytext=(len(ok) + 0.1, 8.0),
                fontsize=8, color=SUAVE, ha="center", va="top",
                arrowprops=dict(arrowstyle="-", color=MUDO, lw=0.8,
                                shrinkA=2, shrinkB=4))
    etiquetas_x = etiquetas + [f"{fallo['num_ctx']//1024}K"]
else:
    etiquetas_x = etiquetas
ax.set_xticks(range(len(etiquetas_x)), etiquetas_x)
ax.set_xlim(-0.75, len(etiquetas_x) - 0.25)
ax.set_ylim(0, 16.5)
ax.grid(axis="y", zorder=0)
ax.legend(frameon=False, loc="upper left", fontsize=8, labelcolor=SUAVE,
          handlelength=1.2, borderpad=0)

# --- 2. La velocidad, que es el techo que no avisa --------------------------
ax = ejes[0][1]
ax.set_title("Velocidad de generación, tokens/s")
tps = [f["tokens_por_s"] for f in ok]
ax.plot(x, tps, color=AZUL, marker="o", ms=5, zorder=3)
for i in (0, len(ok) - 1):
    ax.annotate(f"{tps[i]:.1f}", (i, tps[i]), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(x, etiquetas)
ax.set_ylim(0, max(tps) * 1.3)
ax.grid(axis="y", zorder=0)

# --- 3. Lo que explica la grafica de al lado --------------------------------
ax = ejes[1][0]
ax.set_title("Capas en la GPU, de 33")
capas = [f.get("capas_en_gpu") or 0 for f in ok]
ax.bar(x, capas, 0.6, color=AZUL, zorder=3)
for i, c in enumerate(capas):
    if c and (i == 0 or c != capas[i - 1]):
        ax.annotate(f"{c}", (i, c), textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=9, color=TINTA)
ax.set_xticks(x, etiquetas)
ax.set_ylim(0, 38)
ax.grid(axis="y", zorder=0)

# --- 4. Cuanto tarda en estar listo -----------------------------------------
ax = ejes[1][1]
ax.set_title("Tiempo de carga, segundos")
carga = [f["carga_s"] for f in ok]
ax.bar(x, carga, 0.6, color=AZUL, zorder=3)
peor = max(x, key=lambda i: carga[i])
ax.annotate(f"{carga[peor]:.0f} s", (peor, carga[peor]), textcoords="offset points",
            xytext=(0, 4), ha="center", fontsize=9, color=TINTA)
ax.set_xticks(x, etiquetas)
ax.set_ylim(0, max(carga) * 1.28)
ax.grid(axis="y", zorder=0)

for ax in (ejes[0][0], ejes[0][1], ejes[1][0], ejes[1][1]):
    ax.set_axisbelow(True)
for ax in ejes[1]:                      # la etiqueta del eje, solo abajo
    ax.set_xlabel("ventana pedida (tokens)", fontsize=8)
fig.subplots_adjust(top=0.85, bottom=0.09, hspace=0.33, wspace=0.17)
plt.show()


## Lo que dicen las cuatro

**El techo del modelo no es el techo.** Declara 262 144 y en esta máquina el
último que carga es **131 072 — la mitad exacta**. Y el de 262 144 no falla por
la GPU: falla con este mensaje, que solo aparece en el log de Ollama.

```
Load failed  error="model requires more system memory (12.1 GiB)
                    than is available (10.7 GiB)"
```

**Memoria del sistema, no VRAM.** Aquí está la trampa que no se adivina: el log,
unas líneas antes, dice cuánta RAM cree que hay.

```
msg="system memory" total="7.8 GiB" free="6.9 GiB" free_swap="3.8 GiB"
```

**7.8 GiB**, en una laptop con **31 GB**. No es la RAM de la máquina: es la que
Docker Desktop le dio a su VM de WSL2. El techo de este laboratorio no lo puso
la GPU de 8 GB ni la RAM de 31 GB — **lo puso un archivo de configuración de
Docker**. Los 10.7 GiB "disponibles" del error son esos 6.9 libres más 3.8 de
swap.

**El reparto se mueve, y nunca al revés.** Hasta 32 768 el azul crece y el
naranja no se mueve: el KV cache entra en la VRAM que sobraba, y las 33 capas
siguen en la GPU. En 65 536 el azul toca el techo de la tarjeta y empieza a
crecer el naranja. En 131 072 el azul **baja** —de 7.85 a 7.35 GB— mientras el
naranja se duplica: Ollama sacó capas de la GPU para hacerle lugar al cache.

**Y de ahí sale la curva que importa.** Las capas en la GPU van 32, 32, 32, 32,
27, **16**; los tokens por segundo van 28.5, 28.5, 27.0, 26.9, 14.6, **10.0**.
Es la misma curva dos veces. Multiplicar la ventana por 32 divide la velocidad
por **2.9** y triplica el tiempo de carga —de 8.8 a 23.9 segundos—, y **nada en
la respuesta de la API lo dice**: no hay advertencia ni campo que avise, solo un
modelo que tarda tres veces más.

> **Una corrida por escalón no alcanza.** En una corrida anterior, 16 384 marcó
> 22.5 tok/s —un bache— con las 33 capas intactas, y en esta corrió a 27.0 sin
> bache. Era ruido de la laptop, no una señal. Para publicar cualquiera de estos
> números habría que repetir cada escalón tres veces y tomar la mediana; lo que
> sí aguanta una sola corrida es la **forma**: el acantilado en 65 536 y 131 072
> apareció idéntico las dos veces.

### El KV cache cuesta lo que cuesta, y no lo que dice la fórmula

La fórmula de libro —`2 × capas × cabezas_kv × dim × bytes × tokens`— con los
números que publica la ficha de este modelo da **0.5 MiB por token**. Lo medido,
de la pendiente de la tabla, es **~50 KiB**: **diez veces menos**.

La diferencia no es error de medición. Es que la ficha no publica lo que hace
falta: `head_count_kv` no viene en el archivo —el modelo usa atención agrupada,
así que las cabezas de K y V son muchas menos que las 16 de atención—, y Ollama
levantó el runner con `FlashAttention:Enabled`, que cambia cómo se guarda el
cache. **La fórmula necesita datos que el modelo no publica; la pendiente de la
tabla, no.** Es el argumento del repositorio otra vez: medir sale más barato que
deducir.

## Qué significa esto para el taller

**Se puede subir la ventana, y hay razones para no hacerlo.** Los 4 096 por
omisión no son una limitación arbitraria de Ollama: son el punto donde este
modelo corre con las 33 capas en la GPU en una laptop de 8 GB. Subir a **32 768
es casi gratis** —el KV cache cabe en la VRAM que sobraba, 26.9 contra 28.5
tok/s— y de ahí para arriba se paga en latencia, que con veinte personas
compartiendo la GPU es justo lo que no se puede pagar.

Así que la consigna del taller se queda como está: **historiales cortos y
esquemas de tools planos**, no `num_ctx` más grande. Pero ahora con un número
detrás, y con un matiz: **si alguna vez hace falta más contexto, 32 768 está a
mano y cuesta un 6% de velocidad.** Es bueno saberlo antes de rediseñar un
agente para que quepa en 4 096.

**Cómo subirlo:**

| dónde | cómo | alcance |
|---|---|---|
| una petición | `"options": {"num_ctx": 32768}` | solo esa llamada |
| el servidor | `OLLAMA_CONTEXT_LENGTH=32768` | todo lo que cargue después |
| la VM de Docker | `%UserProfile%\.wslconfig` → `memory=24GB` | sube el techo duro |

El tercero es el que destrabaría el fallo de 262 144, y es el menos obvio: nada
en el error de Ollama menciona Docker. Después de editarlo hay que reiniciar la
VM (`wsl --shutdown`) y Docker Desktop.

## Tu turno

- Repite un escalón tres veces y toma la mediana. ¿Cuánto se mueve?
- Corre el mismo laboratorio con `mistral:latest`, que es más chico. ¿Dónde
  queda su techo, y cuántas capas aguanta en la GPU?
- Sube `memory` en `.wslconfig`, reinicia y vuelve a pedir 262 144. ¿Carga? ¿Y a
  cuántos tokens por segundo?
- Mide la mitad que este laboratorio no toca: **leer un prompt largo**. Aquí
  todos los prompts fueron de ~30 tokens, así que `prompt_eval` no pesó nada. Con
  100 000 tokens de contexto lleno es lo que domina la espera, y se mide con el
  `prompt_eval_duration` que ya devuelve cada llamada.